<a href="https://colab.research.google.com/github/Arda-Avci/AI-Publisher/blob/main/Google_Colab_AI_Publisher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 AI Publisher - Otonom Video Üretim ve Pazarlama Sunucusu

Bu notebook, **AI Publisher** projesi için Google Colab üzerinde çalışan GPU destekli yapay zekâ sunucusunu (CogVideoX-5b, XTTS-v2, AudioLDM2, GFPGAN/RealESRGAN ve Wav2Lip dudak senkronizasyonu) kurar ve başlatır.

### 💡 Kurulum ve Başlatma Talimatı:
1. Üst menüden **Düzenle (Edit) > Defter Ayarları (Notebook Settings)** seçeneğine gidin.
2. Donanım ivmelendiriciyi **T4 GPU** (veya daha üstü bir ekran kartı) olarak seçip kaydedin.
3. Aşağıdaki kod hücresini çalıştırın. Bağımlılıklar ilk kez kurulurken kernel otomatik olarak yeniden başlatılacaktır (Oturum çöktü uyarısı normaldir).
4. Kernel otomatik kapandıktan sonra **aynı hücreyi ikinci kez çalıştırın**.
5. Ngrok Auth Token istendiğinde ngrok.com adresinden alacağınız token'ınızı girin.
6. Sunucu başarıyla ayağa kalktığında size verilen tünel URL'sini kopyalayıp yerel Node.js projenizin `.env` dosyasındaki `COLAB_URL` alanına yapıştırın.

In [ ]:
import os, subprocess, shutil
from google.colab import userdata

repo_dir = "/content/AI-Publisher"

if os.path.exists(repo_dir):
    print("Deleting existing repository directory...")
    shutil.rmtree(repo_dir, ignore_errors=True)

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    GITHUB_TOKEN = input("GitHub token: ").strip()

repo_url = f"https://{GITHUB_TOKEN}@github.com/Arda-Avci/AI-Publisher.git"

print("Repository klonlanıyor...")
subprocess.run(["git", "clone", repo_url, repo_dir], check=True)

print("colab_setup.py kopyalanıyor...")
shutil.copy(os.path.join(repo_dir, "colab_setup.py"), "/content/colab_setup.py")
print("colab_server.py kopyalanıyor...")
shutil.copy(os.path.join(repo_dir, "colab_server.py"), "/content/colab_server.py")

print("Kurulum betiği çalıştırılıyor...")
try:
    subprocess.run(["python3", "-u", "/content/colab_setup.py"], check=True)
except subprocess.CalledProcessError as e:
    # Alt süreç SIGKILL (9) ile sonlandıysa (o.kill(os.getpid(), 9)), ana kernel'i yeniden başlatmak üzere kendimizi öldürüyoruz
    # returncode negatif değer alabilir (örneğin -9) veya shell'e göre 137 veya -15 olabilir.
    # Bu durumda her koşulda süreci yeniden başlatıyoruz.
    print(f"[INFO] Kurulum betiği sonlandı, çıkış kodu: {e.returncode}. Oturum yenileniyor...")
    import os
    os.kill(os.getpid(), 9)